# Retrieval augmented generation (RAG)

## Loading Documents
A first step in RAG is to load document. You need a loader that supports the document type you are interested in. We use in this example Langchain, because it includes a collection of 60+ libraries for multiple types of documents and formats.


A first example with the `PyPDFLoader` library. Pdf support is direct and a single command is enough.

In [1]:
# For this loading Documents part, you may need these packages installed

#!pip install -q langchain
#!pip install -U -q langchain-community
#!pip install --upgrade pip
import sys
print(sys.executable)

import langchain
import langchain_core
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("splitter class loaded OK")

/Users/jerhenry/Documents/Workdoc/f-Virtual_Machines/Environments/Pytorch2026/bin/python3.11
langchain: 1.2.10
langchain_core: 1.2.13
splitter class loaded OK


In [2]:
import warnings # optional, disabling warnings about versions and others
warnings.filterwarnings('ignore') # optional, disabling warnings about versions and others

#!pip install -q pypdf 

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
loader = PyPDFLoader("docs/War-of-the-Worlds.pdf")
book = loader.load()

In [3]:
# How long is the document we loaded?
len(book)

128

In [4]:
#Looking at a small extract, one page, and a few hundred characters in that page
page = book[5]
print(page.page_content[0:500])

darkness were Ottershaw and Chertsey and a ll their hundreds of people, sleeping in 
peace.  
   He was full of speculation that night a bout the condition of Mars, and scoffed at the 
vulgar idea of its having in- habitants w ho were signalling us. His idea was that 
meteorites might be falling in a heavy shower upon the planet, or that a huge volcanic 
explosion was in progress. He pointed out to me how unlikely it was that organic 
evolution had taken the same direction in the two adjacent pl


In [5]:
#Which page is it, from which document?
page.metadata

{'producer': 'PDFill: Free PDF Writer and Tools',
 'creator': 'PyPDF',
 'creationdate': '2011-08-24T10:49:19-04:00',
 'moddate': '2011-08-24T10:49:19-04:00',
 'source': 'docs/War-of-the-Worlds.pdf',
 'total_pages': 128,
 'page': 5,
 'page_label': '6'}

LangChain is of course not the only game in town. Other libraries, like LlamaIndex, can performt the same tasks:

In [6]:
# Loading a simple document with LlamaIndex, also straightforward

from llama_index.core import Document
from pypdf import PdfReader

pdf_path = "docs/War-of-the-Worlds.pdf"
reader = PdfReader(pdf_path)

documents = []

# Loop through pages and create a Document for each
for i, page in enumerate(reader.pages):
    page_text = page.extract_text()
    
    # Create the Document with a metadata dictionary
    doc = Document(
        text=page_text,
        metadata={
            "page_label": i + 1,
            "file_name": "War-of-the-Worlds.pdf"
        }
    )
    documents.append(doc)




In [7]:
# 1. Access the 6th document (page 6)
# Equivalent to: page = book[5]
doc = documents[5]

# 2. Print the first 500 characters
# Equivalent to: print(page.page_content[0:500])
print(doc.text[0:500])

darkness were Ottershaw and Chertsey and a ll their hundreds of people, sleeping in 
peace.  
   He was full of speculation that night a bout the condition of Mars, and scoffed at the 
vulgar idea of its having in- habitants w ho were signalling us. His idea was that 
meteorites might be falling in a heavy shower upon the planet, or that a huge volcanic 
explosion was in progress. He pointed out to me how unlikely it was that organic 
evolution had taken the same direction in the two adjacent pl


In [8]:
# To access the metadata (equivalent to page.metadata in LangChain):
print(documents[5].metadata)
# Output: {'page_label': 6, 'file_name': 'War-of-the-Worlds.pdf'}


{'page_label': 6, 'file_name': 'War-of-the-Worlds.pdf'}


A second example with a Youtube video. We will continue with LangChain to avoid repeats. There is a little more work here. The yt_dlp library will need options to know what audio format to download (we won't care much about the video part). Here we use m4a, at 192 kbps. Then the ffmpeg and ffprobe programs will isolate and stream the audio part. We will then use the OpenAI whisper library to convert the audio into text (speech-to-text).

In [9]:
#!pip install --upgrade --no-deps --force-reinstall -q yt_dlp
#!pip install -q pydub
#!pip install -q ffmpeg
#!pip install -q ffprobe
#!pip install -q more-itertools
#!pip install -q numba
#!pip install --upgrade --no-deps --force-reinstall -q git+https://github.com/openai/whisper.git
#!pip install -q tiktoken

import os
import whisper
from yt_dlp import YoutubeDL
from datetime import datetime

start_time = datetime.now()

# Step 1: Configuration
url = "https://www.youtube.com/watch?v=NVZXFK5aKlQ"
save_dir = "docs/youtube/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': os.path.join(save_dir, '%(title)s.%(ext)s'),
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'm4a',
        'preferredquality': '192',
    }],
    'ffmpeg_location': '/opt/homebrew/bin/ffmpeg', 
}

# Step 2: Check if the file already exists before downloading
with YoutubeDL(ydl_opts) as ydl:
    # Extract info only (doesn't download) to get the expected filename
    info_dict = ydl.extract_info(url, download=False)
    video_title = info_dict.get('title', None)
    
    # Construct the expected local path (yt-dlp sanitizes filenames, so we get the exact one)
    # We change the extension to .m4a because that is our target format
    expected_filename = ydl.prepare_filename(info_dict).rsplit('.', 1)[0] + ".m4a"
    
    if os.path.exists(expected_filename):
        print(f"Found existing file: {expected_filename}. Skipping download.")
        downloaded_file_path = expected_filename
    else:
        print(f"File not found. Downloading: {video_title}...")
        ydl.download([url])
        downloaded_file_path = expected_filename

# Step 3: Load the Whisper model
print("Loading Whisper model...")
model = whisper.load_model("base")

# Step 4: Transcribe the audio file
print(f"Transcribing: {downloaded_file_path}")
result = model.transcribe(downloaded_file_path)

# Optional: Print a snippet of the transcription to verify
print("-" * 30)
print("Transcription Preview:", result['text'][:500])
print("-" * 30)

end_time = datetime.now()
print("Total Duration:", end_time - start_time)

[youtube] Extracting URL: https://www.youtube.com/watch?v=NVZXFK5aKlQ
[youtube] NVZXFK5aKlQ: Downloading webpage


[youtube] NVZXFK5aKlQ: Downloading android vr player API JSON
Found existing file: docs/youtube/The War of the Worlds Summary - H. G. Wells.m4a. Skipping download.
Loading Whisper model...
Transcribing: docs/youtube/The War of the Worlds Summary - H. G. Wells.m4a
------------------------------
Transcription Preview:  The War of the Worlds begins with astronomers observing unusual explosions on Mars. Over several years, these phenomena increase, culminating in the arrival of a cylindrical object that crashes into horse-al-common, near Woking, England. The narrator, a philosophical writer, witnesses the arrival and the subsequent emergence of the Martians. The initial reactions are a mix of curiosity and skepticism. As the cylinder opens, strange, tentacled creatures emerge, followed by the assembly of a mass
------------------------------
Total Duration: 0:00:10.043288


In [10]:
# Adding metadata to the transcript, and saving the transcript to a file so we can use it outside of this program.
class Document:
    def __init__(self, source, text, metadata=None):
        self.source = source
        self.page_content = text
        self.metadata = metadata or {}

# Wrap the transcription result in the Document class with metadata
document = Document(
    source=downloaded_file_path,
    text=result['text'], 
    metadata={"source": "youtube", "file_path": downloaded_file_path}
)
#Save the transcript to a text file
transcript_file_path = os.path.join(save_dir, 'transcript.txt')
with open(transcript_file_path, 'w') as f:
    f.write(result['text'])

print(f"Transcript saved to {transcript_file_path}")

Transcript saved to docs/youtube/transcript.txt


In [11]:
# how many characters in this transcript file?
len(document.page_content)

4998

In [12]:
# Print the first 500 characters of the transcript
print(document.page_content[:500])


 The War of the Worlds begins with astronomers observing unusual explosions on Mars. Over several years, these phenomena increase, culminating in the arrival of a cylindrical object that crashes into horse-al-common, near Woking, England. The narrator, a philosophical writer, witnesses the arrival and the subsequent emergence of the Martians. The initial reactions are a mix of curiosity and skepticism. As the cylinder opens, strange, tentacled creatures emerge, followed by the assembly of a mass


## Splitting our documents in chunks
A second step is to split our documents (a 128-page book and 32K-character trasncript file) into smaller chunks. We use Langchain libraries here again.

In [13]:
# We will use the most important library, recursive character splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
# Chunks have a character length, and an overlap values. For example (in real life, you are probably closer to 500 to 1000 and 50 to 100 respectively):
rsplit = RecursiveCharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=5,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

In [15]:
# Let's take an example string
text1 = 'abcdefghijklmnopqrstuvwxyz1234567890'

In [16]:
# WHat happens if we split it with this splitter?
rsplit.split_text(text1)

['abcdefghijklmnopqrst', 'pqrstuvwxyz123456789', '567890']

In [17]:
# Let's take another example, this time a real text:
Hamlet = """Truly to speak, and with no addition, \
We go to gain a little patch of ground \
That hath in it no profit but the name. \
To pay five ducats, five, I would not farm it; \
Nor will it yield to Norway or the Pole \
A ranker rate, should it be sold in fee."""

In [18]:
# How does the splitter split the Hamlet text?
rsplit.split_text(Hamlet)

['Truly to speak, and',
 'and with no',
 'no addition, We go',
 'go to gain a little',
 'patch of ground',
 'That hath in it no',
 'no profit but the',
 'the name. To pay',
 'pay five ducats,',
 'five, I would not',
 'not farm it; Nor',
 'Nor will it yield',
 'to Norway or the',
 'the Pole A ranker',
 'rate, should it be',
 'be sold in fee.']

In [19]:
# Note that there are many splitter libraries, some are smart, like this recursive character splitter, because it allows you to define different possible bounderies between chunks. Others are more basic, like this one:
# Comparing 2 libraries 
from langchain_text_splitters import CharacterTextSplitter

chunk_size =20
chunk_overlap = 5

csplit = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)


In [20]:
# Remember how the recursive character splitter chopped the long text1 string?
rsplit.split_text(text1)

['abcdefghijklmnopqrst', 'pqrstuvwxyz123456789', '567890']

In [21]:
# Character splitter does not do anything, because it considers by default the end of paragraph as the separator.
csplit.split_text(text1)

['abcdefghijklmnopqrstuvwxyz1234567890']

In [22]:
# You would see the same type of rigid behavior on the Hamlet text: one chunk only, despite our "20 characters max" instruction, 
# because it waits for the end of the paragraph
csplit.split_text(Hamlet)

['Truly to speak, and with no addition, We go to gain a little patch of ground That hath in it no profit but the name. To pay five ducats, five, I would not farm it; Nor will it yield to Norway or the Pole A ranker rate, should it be sold in fee.']

In [23]:
# Let's go back to the recurseive splitter, and go for a more realistic chunk size
rsplit = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

In [24]:
# If we split the pdf document (the book), how many chunks do we get? Remember, the book had 128 pages

len(book)

128

In [25]:
# Splitting by groups of 500 chracters gives us close to 1000 chunks:
rdoc1 = rsplit.split_documents(book)
len(rdoc1)

952

In [26]:
#Let's look at a few chunks:
for i, doc in enumerate(rdoc1[30:33]):  # Adjust the number 3 to print more or fewer splits
    print(f"--- Split {i + 1} ---")
    print(doc.page_content)
    print()  # Print an empty line for better readability


--- Split 1 ---
but that was simply that my eye was tired. Forty millions of miles it was from us--more 
than forty millions of miles of void. Few people realise the im- mensity of vacancy in 
which the dust of the material universe swims.  
   Near it in the field, I re member, were three faint points of  light, three telescopic stars 
infinitely remote, and all around it was th e unfathomable darkness of empty space. You

--- Split 2 ---
infinitely remote, and all around it was th e unfathomable darkness of empty space. You 
know how that blackness looks on a frosty st arlight night. In a tele- scope it seems far 
profounder. And invisible to me because it wa s so remote and small, flying swiftly and 
steadily towards me across that incredible di stance, drawing nearer every min- ute by so 
many thousands of miles, came the Thing they  were sending us, the Thing that was to

--- Split 3 ---
many thousands of miles, came the Thing they  were sending us, the Thing that was to 
bring so

Is 500 characters the right length for our chunks? This is a difficult question, because it depends on the data source, and the writing style (long sentences / short sentences etc.) Let's experiement on an extract of the book:

In [27]:
# Let's consider an extract from the book:
text3 = """ The Martians seem to have calculated their descent with amazing subtlety--their mathematical learning is evidently far in excess of ours--and to have carried out their prepara- tions with a well-nigh perfect unanimity. Had our instru- ments 
permitted it, we might have seen the gathering trouble far back in the nineteenth century. Men like Schiaparelli watched the red planet--it is odd, by-the-bye, that for count- less centuries Mars has been the star of war--but failed to interpret 
the fluctuating appearances of the markings they mapped so well. All that time the Martians must have been getting ready.
During the opposition of 1894 a great light was seen on the illuminated part of the disk, first at the Lick Observatory, then by Perrotin of Nice, and then by other observers. English readers heard of it first in the issue of NATURE dated August 2. I am 
inclined to think that this blaze may have been the casting of the huge gun, in the vast pit sunk into their planet, from which their shots were fired at us. Peculiar markings, as yet unexplained, were seen near the site of that outbreak during the next 
two oppositions.
The storm burst upon us six years ago now. As Mars approached opposition, Lavelle of Java set the wires of the astronomical exchange palpitating with the amazing intelli- gence of a huge outbreak of incandescent gas upon the planet. It had occurred towards 
midnight of the twelfth; and the spectroscope, to which he had at once resorted, indicated a mass of flaming gas, chiefly hydrogen, moving with an enormous velocity towards this earth. This jet of fire had become invisible about a quarter past twelve. He 
compared it to a colossal puff of flame suddenly and violently squirted out of the planet, as flaming gases rushed out of a gun. """

In [28]:
# Let's start higher than 500 characters. If our target is 700 characters max with a very small overlap, how many chunks would we get with this text, and how would they look like?
chunk_size =700
chunk_overlap = 5
rsplit = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

chunks=rsplit.split_text(text3)
for i, _ in enumerate(chunks):
    print(f"chunk # {i}, size: {len(chunks[i])}")
print(chunks[0])

chunk # 0, size: 610
chunk # 1, size: 526
chunk # 2, size: 642
The Martians seem to have calculated their descent with amazing subtlety--their mathematical learning is evidently far in excess of ours--and to have carried out their prepara- tions with a well-nigh perfect unanimity. Had our instru- ments 
permitted it, we might have seen the gathering trouble far back in the nineteenth century. Men like Schiaparelli watched the red planet--it is odd, by-the-bye, that for count- less centuries Mars has been the star of war--but failed to interpret 
the fluctuating appearances of the markings they mapped so well. All that time the Martians must have been getting ready.


In [29]:
# What happens if the split is smaller:
chunk_size =300
chunk_overlap = 5
rsplit = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
chunks=rsplit.split_text(text3)
for i, _ in enumerate(chunks):
    print(f"chunk # {i}, size: {len(chunks[i])}")
print(chunks[0])

chunk # 0, size: 240
chunk # 1, size: 245
chunk # 2, size: 121
chunk # 3, size: 253
chunk # 4, size: 271
chunk # 5, size: 256
chunk # 6, size: 254
chunk # 7, size: 128
The Martians seem to have calculated their descent with amazing subtlety--their mathematical learning is evidently far in excess of ours--and to have carried out their prepara- tions with a well-nigh perfect unanimity. Had our instru- ments


In [30]:
# A chnk of 700 gives us a few sentences (probably too long), 300 characters seem to fit one sentence. Looks optimal, but can we go smaller?
chunk_size =100
chunk_overlap = 5
rsplit = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
chunks=rsplit.split_text(text3)
for i, _ in enumerate(chunks):
    print(f"chunk # {i}, size: {len(chunks[i])}")
print(chunks[0])

chunk # 0, size: 92
chunk # 1, size: 96
chunk # 2, size: 52
chunk # 3, size: 99
chunk # 4, size: 95
chunk # 5, size: 54
chunk # 6, size: 96
chunk # 7, size: 29
chunk # 8, size: 98
chunk # 9, size: 95
chunk # 10, size: 61
chunk # 11, size: 97
chunk # 12, size: 90
chunk # 13, size: 73
chunk # 14, size: 16
chunk # 15, size: 97
chunk # 16, size: 92
chunk # 17, size: 74
chunk # 18, size: 97
chunk # 19, size: 96
chunk # 20, size: 69
chunk # 21, size: 93
chunk # 22, size: 37
The Martians seem to have calculated their descent with amazing subtlety--their mathematical


## Storing in Vector Store
The third step is to store your splits in a vector database. There are dozens of solutions. Very popular solutions for local storage include Mongodb, Chroma, Weaviate and Milvus. All large Cloud vendors (Azure, AWS etc.) offer a Cloud vectordb solution. Here we use Chroma, a locally stored, flexible popular choice. 

Before storing our data into the vectordb, we need to convert the text strings into vectors (embedding). We use a tokenizer compatible with the BERT model to first tokenize the text, then embed (convert to vectors).

In [31]:
# Create Ollama embeddings
start_time = datetime.now()
#!pip install -q chromadb

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
all_splits = rdoc1
embeddings = OllamaEmbeddings(model="nomic-embed-text")

end_time = datetime.now()

print("Duration:", end_time - start_time)

Duration: 0:00:00.026647


What does an embedding, or a "vector", look like in practice? Let's take a few sentences and look at the vectors and their similarity

In [32]:
text1 = "i like hotdogs"
text2 = "i like sandwiches"
text3 = "this is a large building"

In [33]:
# Let's embed these 3 very interesting sentences:
embedding1 = embeddings.embed_query(text1)
embedding2 = embeddings.embed_query(text2)
embedding3 = embeddings.embed_query(text3)

In [34]:
# Let's look at the vector for embedding 1, for the first sentence... or at least the first 10 numbers, as the vector has 768 components
# Change :10 to a larger number if you want to see more components
print("embedding1 includes", len(embedding1), "values")
print("First few values:", embedding1[:10])

embedding1 includes 768 values
First few values: [-0.1991678923368454, -0.021470991894602776, -3.722862720489502, -0.7221522331237793, 0.05586780607700348, 0.9450611472129822, -1.1482144594192505, 0.5535275340080261, -0.9894599914550781, -0.8412603735923767]


How closes are these vectors from one another? There are many ways to compare them, here we use the cosine similarity method.

In [35]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
# Step 1 : creating the normalized vectors (so the product is between 0 and 1)

norm_a = np.linalg.norm(embedding1)
norm_b = np.linalg.norm(embedding2)
norm_c = np.linalg.norm(embedding3)
normalized_a = embedding1 / norm_a
normalized_b = embedding2 / norm_b
normalized_c = embedding3 / norm_c

#Step 2: comparing text1 and text 2 embeddings, then text1 and text 3 embeddings:

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

similarity_1_2 = cosine_similarity(embedding1, embedding2)
similarity_1_3 = cosine_similarity(embedding1, embedding3)

print("Similarity (with cos similarity) between sentence 1 and 2:", similarity_1_2)
print("Similarity (with cos similarity) between sentence 1 and 3:", similarity_1_3)

Similarity (with cos similarity) between sentence 1 and 2: 0.717907398962642
Similarity (with cos similarity) between sentence 1 and 3: 0.37678030184616623


Sentences 1 and 2 are closer to each other than sentences 1 and 3

Now that we have embeddings, let's store them into a vector database (we use Chroma).

In [36]:
#!pip install --upgrade langchain chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# choose an embeddings backend you actually use
from langchain_community.embeddings import OllamaEmbeddings  # or HuggingFaceEmbeddings


# Set the environment variable to disable tokenizers parallelism and avoid warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Let's define a directory where we'll store the database beyond this notebook execution (and let's make sure it is emtpy, as I run this notebook often :))
persist_directory = 'docs/chroma/'
!rm -rf ./docs/chroma  # remove old database files if any

In [37]:
# Now that we have the structyure, let's store the vectors and matching text segments
start_time = datetime.now()

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="war_of_the_worlds"
)

end_time = datetime.now()

print("Duration:", end_time - start_time)

Duration: 0:00:42.485667


Now let's see if we can perform some similarity search with this database. keep in mind that we are just comparing vectors here, there is no LLM yet to smartly correlate deeper.

In [38]:
#let's define a question, so we have "words" to run a similarity search against:
question = "Did the spaceship come from the planet Mars?"

In [39]:
#Let's retrieve the best (closest) 5 chunks from the Chroma database, i.e. those which words have highest simiarity to toe words of the question:
docs = vectordb.similarity_search(question,k=5)

In [40]:
# We retrieved 5 segments:
len(docs)

5

In [41]:
# Let's look at one of these segments:
docs[0].page_content

'own warmer planet, green with vegetation and grey with water, with a cloudy atmosphere \neloquent of fertility, with glimpses through its drifting cloud wisps of  broad stretches of \npopulous country and narrow, navy-crowded seas.  \n   And we men, the creatures who inhabit this earth, must be to them at least as alien and \nlowly as are the monkeys and lemurs to us. The intellectual side of man already admits'

All 5 chunks will  have some words similar to that of the query. But there are different ways of determining similarity. The classical cos (or sin) similarity algorithm selects the chunks in the vector database that are closes to the query (contains the largest number of common words), but this is not always the best choice. Consider these 3 sentences for example:

In [42]:
text4 = [
    """The alien spaceships looked like flying saucers.""",
    """The alien spaceships were round in shape.""",
    """The spaceships were destroying everything.""",
]

In [43]:
# Let's create a mini-vector database from these 3 sentences:
tempdb_mini = Chroma.from_texts(text4, embedding=embeddings, collection_name="mini_test" )

In [44]:
# What happens if we ask about the best 2 segments matching the question:
question = "What can you tell me about the alien spaceships?"
tempdb_mini.similarity_search(question, k=2)

[Document(metadata={}, page_content='The alien spaceships were round in shape.'),
 Document(metadata={}, page_content='The alien spaceships looked like flying saucers.')]

The 2 segments are those that contain the larges word overlap with the quesiton, namely 'alien' and 'spaceship'. The third chunk is not selected because it has only one overlapping word (spaceships). ... but it does contain useful information (they are destroying everything!)
Similarity search points to the documents that are closest semantically to the question, which may include a lot of redundant information, and miss some key points, for example that the alien spaceships were destroying everything. Max Marginal Relevance (MMR) search improves Similarity Search by picking the top k as Similarity Search does, but returning the vectors that are farthest from each other (in this top k list), so as to maximize the diversity of information returned.

In [45]:
tempdb_mini.max_marginal_relevance_search(question, k=2)

[Document(metadata={}, page_content='The alien spaceships were round in shape.'),
 Document(metadata={}, page_content='The spaceships were destroying everything.')]

## Retrieving with the LLM in action
The full process consists of asking a question, retrieving the relevant information, then passing the information and the question to the LLM.

In [46]:
#Prerequisite, you need an LLM, here I ma using Ollama a,d lalam3.1:latest
#!pip install -q ollama
#!ollama serve & ollama pull llama3.1:latest & ollama pull nomic-embed-text

In [47]:
# Let's define a question:
question = "Did the spaceship come from the planet Mars?"

In [48]:
#Let's first ask the question to the LLM WITHOUT RAG, so the LLM only has its own knowledge to respond:
start_time = datetime.now()

from langchain_community.llms import Ollama
llm = Ollama(model = "llama3.1:latest")
response = llm.invoke("Are there aliens on Mars?")
end_time = datetime.now()

print("Duration:", end_time - start_time)
print("Response:", response)

Duration: 0:00:40.747324
Response: While there's currently no definitive evidence of aliens on Mars, there are many reasons to believe that the Red Planet is a prime candidate for supporting life. Here's a summary of the current state of knowledge:

**Mars Exploration:**

NASA's Curiosity rover, which has been exploring Mars since 2012, has provided a wealth of information about the planet's geology, climate, and potential habitability. The rover has discovered evidence of ancient lakes, rivers, and even an ocean on Mars, which suggests that the planet may have been capable of supporting life in the past.

**Biosignatures:**

In 2020, NASA's Perseverance rover began exploring Mars, and it's equipped with instruments designed to search for signs of past or present life on the planet. The rover is carrying a sample collection system that will allow scientists to analyze Martian rocks and soil for biosignatures, such as organic molecules or evidence of ancient microbial life.

**Potential

In [49]:
# Now let's point the LLM to the RAG database, and instruct it to use the data from the RAG system to respond:
import ollama
from IPython.display import Markdown, display

# 1. Core Logic (Kept from your original code)
def ollama_llm(question, context):
    formatted_prompt = f"Question: {question}\n\nContext: {context}"
    response = ollama.chat(
        model='llama3.1:latest', 
        messages=[{'role': 'user', 'content': formatted_prompt}]
    )
    return response['message']['content']

# Define the RAG setup (Ensure vectordb is already defined in your notebook)
retriever = vectordb.as_retriever()

def rag_chain(question):
    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)
    # Combine chunks into one context string
    formatted_context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    # Get answer from LLM
    return ollama_llm(question, formatted_context)

# 2. Execution Block
# You can either hardcode the question or use input() for interactivity
question = "did the aliens eventually go on to land on Venus?" 

print("Processing your request...")
answer = rag_chain(question)

# 3. Display the output nicely below the cell
display(Markdown(f"### Question\n{question}"))
display(Markdown(f"### Answer\n{answer}"))

Processing your request...


### Question
did the aliens eventually go on to land on Venus?

### Answer
According to the text, there is no indication that the aliens (Martians) landed on Venus. In fact, the text states that the Martians "had fired at us" (i.e., the people on Earth) and that the missiles were "rushing now at a pace of many miles a second through the empty gulf of space, hour by hour and day by day, nearer and nearer" to Earth, not Venus.

The text does mention that Lessing has advanced reasons for supposing that the Martians "actually succeeded in effecting a landing on the planet Venus", but this is a speculation or a hypothesis presented by Lessing, and it is not stated as a fact that the Martians actually landed on Venus.

In [52]:
# The itnerface could be nicer, we could use gradio for example, to display a chatbot-like interface:
import gradio as gr
import ollama
from bs4 import BeautifulSoup as bs
from langchain_community.embeddings import OllamaEmbeddings

# Create Ollama embeddings and vector store
#embeddings = OllamaEmbeddings(model="nomic-embed-text")
#vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

# Define the function to call the Ollama Qwen model
def ollama_llm(question, context):
    formatted_prompt = f"Question: {question}\n\nContext: {context}"
    response = ollama.chat(model='llama3.1:latest', messages=[{'role': 'user', 'content': formatted_prompt}])
    return response['message']['content']

# Define the RAG setup
retriever = vectordb.as_retriever()

def rag_chain(question):
    retrieved_docs = retriever.invoke(question)
    formatted_context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return ollama_llm(question, formatted_context)

# Define the Gradio interface
def get_important_facts(question):
    return rag_chain(question)

# Create a Gradio app interface
iface = gr.Interface(
  fn=get_important_facts,
  inputs=gr.Textbox(lines=2, placeholder="Enter your question here..."),
  outputs="text",
  title="RAG with llama3.1",
  description="Ask questions about the provided context",
)

# Launch the Gradio app
iface.launch()
# example q: did the aliens eventually go on to land on Venus?

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
